# 2장 코드 필사

### 코드 2-1 필요한 라이브러리 호출

In [ ]:
import torch
import torch.nn as nn
import numpy as np    # 벡터 및 행렬 연산에서 매우 편리한 기능을 제공하는 파이썬 라이브러리 패키지
import pandas as pd   # 데이터 처리를 위해 널리 사용되는 파이썬 라이브러리 패키지
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### 코드 2-2 데이터 호출

In [ ]:
dataset = pd.read_csv('../chap02/data/car_evaluation.csv')   # ①
dataset.head()   # ②

### 코드 2-3 예제 데이터셋 분포

In [ ]:
fig_size = plt.rcParams["figure.figsize"]
fig_size[0] = 8
fig_size[1] = 6
plt.rcParams["figure.figsize"] = fig_size
dataset.output.value_counts().plot(kind='pie', autopct='%0.05f%%',
colors=['lightblue', 'lightgreen', 'orange', 'pink'], explode=(0.05, 0.05, 0.05, 0.05))

### 코드 2-4 데이터를 범주형 타입으로 변환

In [ ]:
categorical_columns = ['price', 'maint', 'doors', 'persons', 'lug_capacity', 'safety']   # 예제 데이터셋 칼럼들의 목록

for category in categorical_columns:
    dataset[category] = dataset[category].astype('category')   # astype() 메서드를 이용하여 데이터를 범주형으로 변환

price = dataset['price'].cat.codes.values   # ①
maint = dataset['maint'].cat.codes.values
doors = dataset['doors'].cat.codes.values
persons = dataset['persons'].cat.codes.values
lug_capacity = dataset['lug_capacity'].cat.codes.values
safety = dataset['safety'].cat.codes.values

categorical_data = np.stack([price, maint, doors, persons, lug_capacity, safety], 1)   # ②
categorical_data[:10]   # 합친 넘파이 배열 중 열 개의 행을 출력하여 보여 줍니다.

### 코드 2-5 배열을 텐서로 변환

In [ ]:
categorical_data = torch.tensor(categorical_data, dtype=torch.int64)
categorical_data[:10]

### 코드 2-6 레이블로 사용할 칼럼을 텐서로 변환

In [ ]:
outputs = pd.get_dummies(dataset.output)   # ①
outputs = outputs.values
outputs = torch.tensor(outputs).flatten()   # 1차원 텐서로 변환

print(categorical_data.shape)
print(outputs.shape)

### 코드 2-7 범주형 칼럼을 N차원으로 변환

In [ ]:
categorical_column_sizes = [len(dataset[column].cat.categories) for column in
                            categorical_columns]
categorical_embedding_sizes = [(col_size, min(50, (col_size+1)//2)) for col_size in
                               categorical_column_sizes]
print(categorical_embedding_sizes)

### 코드 2-8 데이터셋 분리

In [ ]:
total_records = 1728
test_records = int(total_records * .2)   # 전체 데이터 중 20%를 테스트 용도로 사용

categorical_train_data = categorical_data[:total_records - test_records]
categorical_test_data = categorical_data[total_records - test_records:total_records]
train_outputs = outputs[:total_records - test_records]
test_outputs = outputs[total_records - test_records:total_records]

### 코드 2-9 데이터셋 분리 확인

In [ ]:
print(len(categorical_train_data))
print(len(train_outputs))
print(len(categorical_test_data))
print(len(test_outputs))

### 코드 2-10 모델의 네트워크 생성

In [ ]:
class Model(nn.Module):   # ①
    def __init__(self, embedding_size, output_size, layers, p=0.4):   # ②
        super().__init__()   # ③
        self.all_embeddings = nn.ModuleList([nn.Embedding(ni, nf) for ni,
                                             nf in embedding_size])
        self.embedding_dropout = nn.Dropout(p)

        all_layers = []
        num_categorical_cols = sum((nf for ni, nf in embedding_size))
        input_size = num_categorical_cols   # 입력층의 크기를 찾기 위해 범주형 칼럼 개수를 input_size 변수에 저장

        for i in layers:   # ④
            all_layers.append(nn.Linear(input_size, i))
            all_layers.append(nn.ReLU(inplace=True))
            all_layers.append(nn.BatchNorm1d(i))
            all_layers.append(nn.Dropout(p))
            input_size = i

        all_layers.append(nn.Linear(layers[-1], output_size))
        self.layers = nn.Sequential(*all_layers)   # 신경망의 모든 계층이 순차적으로 실행되도록 모든 계층에 대한 목록(all_layers)을 nn.Sequential 클래스로 전달

    def forward(self, x_categorical):   # ⑤
        embeddings = []
        for i,e in enumerate(self.all_embeddings):
            embeddings.append(e(x_categorical[:,i]))
        x = torch.cat(embeddings, 1)   # 넘파이의 concatenate와 같지만 대상이 텐서가 됩니다.
        x = self.embedding_dropout(x)
        x = self.layers(x)
        return x

### 코드 2-11 Model 클래스의 객체 생성

In [ ]:
model = Model(categorical_embedding_sizes, 4, [200,100,50], p=0.4)
print(model)

### 코드 2-12 모델의 파라미터 정의

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 코드 2-13 CPU/GPU 사용 지정

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')   # GPU가 있다면 GPU를 사용
else:
    device = torch.device('cpu')   # GPU가 없다면 CPU를 사용

### 코드 2-14 모델 학습

In [ ]:
epochs = 500
aggregated_losses = []
train_outputs = train_outputs.to(device=device, dtype=torch.int64)

for i in range(epochs):   # for 문은 500회 반복되며, 각 반복마다 손실 함수가 오차를 계산
    i += 1
    y_pred = model(categorical_train_data).to(device)
    single_loss = loss_function(y_pred, train_outputs)
    aggregated_losses.append(single_loss)   # 반복할 때마다 오차를 aggregated_losses에 추가

    if i%25 == 1:
        print(f'epoch: {i:3} loss: {single_loss.item():10.8f}')

    optimizer.zero_grad()
    single_loss.backward()   # 가중치를 업데이트하기 위해 손실 함수의 backward() 메서드 호출
    optimizer.step()   # 옵티마이저 함수의 step() 메서드를 이용하여 기울기 업데이트

print(f'epoch: {i:3} loss: {single_loss.item():10.10f}')   # 오차가 25 에포크마다 출력

### 코드 2-15 테스트 데이터셋으로 모델 예측

In [ ]:
test_outputs = test_outputs.to(device=device, dtype=torch.int64)
with torch.no_grad():
    y_val = model(categorical_test_data)
    loss = loss_function(y_val, test_outputs)
print(f'Loss: {loss:.8f}')

### 코드 2-16 모델의 예측 확인

In [ ]:
print(y_val[:5])

### 코드 2-17 가장 큰 값을 갖는 인덱스 확인

In [ ]:
y_val = np.argmax(y_val, axis=1)
print(y_val[:5])

### 코드 2-18 테스트 데이터셋을 이용한 정확도 확인

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print(confusion_matrix(test_outputs,y_val))
print(classification_report(test_outputs,y_val))
print(accuracy_score(test_outputs, y_val))